# Entregável 8 — Redução de Dimensionalidade

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Combater a "Maldição da Dimensionalidade" reduzindo o número de features do nosso modelo 
preservando simultaneamente a maior parte de sua energia, estrutura e capacidade preditiva.  
Utilizaremos **PCA (Principal Component Analysis)** para mapear combinações lineares densas 
da matriz esparsa, e visualizaremos os agrupamentos médicos no espaço através da técnica não-linear projetiva **t-SNE**.

### Conteúdo abordado:
- PCA Clássico e cálculo das Variâncias Explicadas (Scree Plot).
- t-SNE (t-Distributed Stochastic Neighbor Embedding) para dispersão visual intuitiva.
- Validação Obrigatória: Teste de Preservação topológica (Correlação de Ranks das Distâncias Euclidianas Originais VS Pós-PCA).

## 1. Configuração e Imports

In [ ]:
!pip install scikit-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

# Configurações de plot
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Diretórios
BASE_DIR = Path('..')
FIG_DIR = Path('figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')

## 2. Carregamento do Dataset Refinado (Entregável 7)

In [ ]:
csv_path = BASE_DIR / 'entregavel-7' / 'dataset_features_engenheirado.csv'
df = pd.read_csv(csv_path)

cols_meta = ['ecg_id', 'derivacao', 'janela_instancia', 'target_is_norm']
cols_feat = [c for c in df.columns if c not in cols_meta]

X = df[cols_feat].values
y = df['target_is_norm'].values

print(f"-> Carregadas {X.shape[0]} amostras com {X.shape[1]} features dimensionais purificadas Z-Score.")
print("Obs: Os dados JÁ ESTÃO normalizados por StandartScaler no Entregável 7, regra Ouro para PCA.")

## 3. Principal Component Analysis (PCA)

In [ ]:
# Rodar o PCA sem limites primeiro para extrair as variâncias totais geradas pelos autovalores
pca_full = PCA()
pca_full.fit(X)

var_ratio = pca_full.explained_variance_ratio_
var_cumulativa = np.cumsum(var_ratio)

# Encontrar o número minimo de componentes pra reter 95% da energia do problema
n_95 = np.argmax(var_cumulativa >= 0.95) + 1  # Indice da primeira vez que bate >= 0.95

fig, ax1 = plt.subplots(figsize=(10, 6))

# Barra da Variância isolada - Scree Plot
ax1.bar(range(1, len(var_ratio)+1), var_ratio, alpha=0.6, color='steelblue', label='Variância Individual (Scree)')
ax1.set_xlabel('Principal Component (PC)')
ax1.set_ylabel('Variância Explicada Ratio', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Linha Continua da Cumulativa
ax2 = ax1.twinx()
ax2.plot(range(1, len(var_ratio)+1), var_cumulativa, 'r*-', linewidth=2, label='Variância Cumulativa')
ax2.axhline(0.95, color='orange', linestyle='--', label='Corte de 95% da Energia Matemátaica')
ax2.axvline(n_95, color='orange', linestyle=':')
ax2.set_ylabel('Soma das Variâncias', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim([0, 1.05])

plt.title(f'PCA Scree Plot\n(Necessários apenas *{n_95} Componentes* para reter 95% de variância de originais {X.shape[1]})', fontweight='bold')

fig.legend(loc='lower center', bbox_to_anchor=(0.5, 0.15), bbox_transform=ax1.transAxes)

plt.tight_layout()
plt.savefig(FIG_DIR / 'pca_scree_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Realizando o corte efetivo das colunas (Aplicando o numero magico descoberto n_95)
pca = PCA(n_components=n_95)
X_pca = pca.fit_transform(X)

print(f"-> Transformação PCA aplicada: Dimensão reduzida radicalmente de {X.shape[1]} colunas gigantes para apenas {n_95} componentes principais matemáticos limpos.")

## 4. Validação Obrigatória: Preservação da Taxonomia Espacial (Rank Correlation)

Se condensarmos 16 dimensões em 8 usando matrizes de rotação, corremos o risco de pacientes originalmente semelhantes sumirem para lados opostos no novo espaço?

Para provar que nosso PCA conservou os "Vizinhos Médicos", medimos distâncias Par-a-Par no universo Original 16D, e depois no PCA 8D. Tiramos o Coeficiente de Kendall/Spearman dos Rankings de quem é próximo a quem.

In [ ]:
print('=' * 80)
print('INICIANDO CÁLCULO DE VALIDAÇÃO ESPACIAL DE MANTEL (Trustworthiness Analógico)')
print('=' * 80)

# Por questões de limite de RAM/tempo numa matriz N² de grandes datasets, pegamos os primeiros ~800 amostras 
amostras_val = min(800, X.shape[0])
X_subset = X[:amostras_val]
X_pca_subset = X_pca[:amostras_val]

# Matriz de distâncias Euclidiana comprimida pdist
dist_originais = pdist(X_subset, metric='euclidean')
dist_pca = pdist(X_pca_subset, metric='euclidean')

# Spearman Rank para saber se quem era vizinho manteve-se vizinho
rho, p = spearmanr(dist_originais, dist_pca)

print(f"-> Testadas sub-matrizes de {len(dist_originais)} combinações de distâncias relativas par-a-par.")
print(f"-> Correlação de Ranking (Rho de Spearman): {rho:.4f}")
if rho > 0.90:
    print("✅ VALIDAÇÃO SUCESSO. A estrutura macro do array foi brutalmente perfeitamente preservada.")
else:
    print("❌ Atenção, houve deformação do espaço.")

## 5. Visualização e Scatter Plots não-lineares com t-SNE

O PCA lida bem em agrupar retas ortogonais no hiperespaço denso, mas para 
plotar distribuições agrupadas no Olho Humano (2D), nada supera a inteligência das projeções manifold, como o t-SNE.

In [ ]:
print('Computando t-SNE 2D no dataset reduzido pelo PCA... (pode demorar alguns segundos)')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_pca)

df_tsne = pd.DataFrame({'Target': y, 'TSNE1': X_tsne[:, 0], 'TSNE2': X_tsne[:, 1]})
df_tsne['Target'] = df_tsne['Target'].map({1: 'Saudáveis (NORM)', 0: 'Doentes (Anormais)'})

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df_tsne,  x="TSNE1", y="TSNE2",
    hue="Target", palette=['crimson', 'mediumseagreen'],
    s=60, alpha=0.7, edgecolor='k'
)

plt.title('t-SNE Dispersão 2D das Instâncias do ECG', fontweight='bold', fontsize=16)
plt.xlabel('Componente t-SNE 1')
plt.ylabel('Componente t-SNE 2')

plt.legend(title='Base de Dados Clínica Rótulos Médicos')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tsne_dispersao_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 6. Salvar Novo Dataset Minimalista

Vamos reconstruir o DataFrame apenas com as colunas de Metadados e os the pseudo-Components (PC1 até PCn) originados para as próximas etapas (Feature Selection do pacote E.9).

In [ ]:
# Montar nomes Dinamicos das Features
nomes_pcs = [f'PC{i+1}' for i in range(n_95)]
df_final_pca = pd.DataFrame(X_pca, columns=nomes_pcs)

# Devolver Metadados inalterados
for col in reversed(cols_meta):
    df_final_pca.insert(0, col, df[col].values)

output_csv = FIG_DIR.parent / 'dataset_features_reduzidas_pca.csv'
df_final_pca.to_csv(output_csv, index=False)

print("=================================================================")
print(f"Dataset REDUZIDO gerado com sucesso para a Arquitetura Super Leve!")
print(f"Caminho: {output_csv}")
print(f"Tamanho Original: {X.shape[1]} Dimensões -> Novo Tamanho: {n_95} Dimensões")
print("=================================================================")

display(df_final_pca.head(3).round(4))